In [1]:
!hostname

gpua070.delta.ncsa.illinois.edu


In [2]:
import scanpy as sc
import scipy.sparse as sp
from pathlib import Path
import numpy as np
import pandas as pd
import anndata as ad
ad.settings.allow_write_nullable_strings = True
import matplotlib.pyplot as plt
import seaborn as sns

In [7]:
import muon as mu
# Import a module with ATAC-seq-related functions
from muon import atac as ac

/projects/bhdw/asachan/.conda/envs/moscot/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/projects/bhdw/asachan/.conda/envs/moscot/lib/python3.11/site-packages/muon/_core/preproc.py:31: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  if Version(scanpy.__version__) < Version("1.10"):


In [6]:
import os
os.chdir('/projects/bhdw/asachan/methods/FIREFate/moscot')
import sys
import logging
import warnings

objects_dir = "/work/hdd/bgdb/asachan/datasets_proj/SKM_ageing_human"
out_tmp = '/projects/bhdw/asachan/tmp'
figures_dir = '/work/hdd/bhdw/asachan/plot_outs/earthDT'

In [5]:
pd.set_option('mode.string_storage', 'python')

## Load preprocessed ATAC and RNA AnnData

Both h5ads are expected to already have raw counts in `.X`, `obs.sample`, `obs.age`, and the embeddings the GW solver consumes (`obsm.X_lsi` for ATAC, `obsm.X_pca` for RNA). The earlier LSI / PCA preprocessing cells have been moved out of this notebook — re-run them upstream once if needed.

In [10]:
atac_file = '/work/hdd/bgdb/asachan/datasets_proj/SKM_ageing_human/atac_objects/atac_fiber/atac_female_type2.h5ad'
adata_atac = sc.read_h5ad(atac_file)

In [11]:
# add age as categorical
adata_atac.obs["age_categorical"] = adata_atac.obs["age"].astype("category")

In [13]:
adata_atac

AnnData object with n_obs × n_vars = 5830 × 27649
    obs: 'replicates', 'TSSEnrichment', 'ReadsInTSS', 'ReadsInPromoter', 'ReadsInBlacklist', 'PromoterRatio', 'PassQC', 'NucleosomeRatio', 'nMultiFrags', 'nMonoFrags', 'nFrags', 'nDiFrags', 'BlacklistRatio', 'orig.ident', 'sample', 'group', 'ReadsInPeaks', 'FRIP', 'fiber_class_1_anno', 'Annotation', 'UMAP_1', 'UMAP_2', 'fiber_class_anno', 'country', 'age', 'Sex', 'age_categorical'
    var: 'n_cells'
    uns: 'Annotation_colors', 'lsi', 'neighbors', 'sample_colors', 'umap'
    obsm: 'X_UMAP', 'X_lsi', 'X_umap'
    varm: 'LSI'
    layers: 'Tn5_insertion_counts'
    obsp: 'connectivities', 'distances'

In [12]:
rna_file = '/work/hdd/bgdb/asachan/datasets_proj/SKM_ageing_human/rna_objects/rna_female_type2_ds_wrt_HALLMARK_DNA_REPAIR.h5ad'
adata_rna = sc.read_h5ad(rna_file)

In [14]:
adata_rna

AnnData object with n_obs × n_vars = 3989 × 48355
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'sample', 'percent.mt', 'age', 'tech', 'Sex', 'Country', 'age_pop', 'Annotation', 'Pseudotime', 'Pseudotime_typeI', 'Pseudotime_typeII', 'bead_count', 'age_categorical', 'GOBP_DNA_DAMAGE_RESPONSE', 'GOBP_DNA_REPAIR', 'HALLMARK_DNA_REPAIR', 'REACTOME_DNA_REPAIR'
    var: 'features', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'hvg', 'neighbors', 'pca', 'rank_genes_groups', 'sample_colors', 'umap'
    obsm: 'X_pca', 'X_umap', 'score_aucell'
    varm: 'PCs'
    layers: 'counts', 'counts_float', 'lognorm'
    obsp: 'connectivities', 'distances'

In [15]:
# Guarantee adata_rna.layers['counts'] is set with raw integer counts.
# Downstream MaxToki RVE tokenization (maxtoki_multimodal_prep.ipynb)
# uses this layer; lognorm or scaled .X would silently mistokenize.
import scipy.sparse as sp

if "counts" not in adata_rna.layers:
    if adata_rna.raw is not None:
        adata_rna.layers["counts"] = adata_rna.raw.X.copy()
        print("hoisted adata_rna.raw.X -> adata_rna.layers['counts']")
    else:
        adata_rna.layers["counts"] = adata_rna.X.copy()
        print("[warn] no .raw and no layers['counts']; copied .X into "
              "layers['counts']. Verify .X holds raw counts (integer-valued).")

C = adata_rna.layers["counts"]
max_val = float(C.data.max()) if sp.issparse(C) else float(C.max())
print(f"adata_rna.layers['counts']: shape={C.shape}  dtype={C.dtype}  "
      f"sparse={sp.issparse(C)}  max={max_val:.1f}")

adata_rna.layers['counts']: shape=(3989, 48355)  dtype=int32  sparse=True  max=3011.0


# Solve quadratic Gromov-Wasserstein OT

If the per-sample artifacts under `/projects/bhdw/asachan/tmp/T_npy_sample_matched/` (`T_*.npy` + `src_obs_*.npy` + `tgt_obs_*.npy`) are already on disk, skip the next three cells (the solver loop and its top-1 / eff-k QC, plus the disk save) and jump straight to the bridge cell at the start of the next section — it loads `T_per_sample` from disk if it isn't already in memory.

In [16]:
samples = sorted(adata_atac.obs["sample"].unique().tolist())
print("samples:", samples)

samples: ['OM6', 'OM9', 'YM2']


In [ ]:
import numpy as np, pathlib
from moscot.problems.cross_modality import TranslationProblem

def solve_one_sample(s, a_s, r_s, eps=1e-3, attempt=1):
    tp_s = TranslationProblem(adata_src=a_s, adata_tgt=r_s)
    tp_s = tp_s.prepare(
        src_attr={"attr": "obsm", "key": "X_lsi"},
        tgt_attr={"attr": "obsm", "key": "X_pca"},
    )
    tp_s = tp_s.solve(
        epsilon       = eps,
        alpha         = 1.0,
        scale_cost    = "mean",
        max_iterations= 10_000,                # ↑ from 2000 (outer GW)
        min_iterations= 50,
        threshold     = 1e-3,
        linear_solver_kwargs={
            "max_iterations": 20_000,          # ↑ from 5000 (inner Sinkhorn)
            "lse_mode": True,
            "threshold": 1e-3,
        },
    )
    (_, prob), = tp_s.problems.items()
    return tp_s, prob

T_per_sample, conv = {}, {}
for s in samples:
    a_s = adata_atac[adata_atac.obs["sample"] == s].copy()
    r_s = adata_rna [adata_rna .obs["sample"] == s].copy()
    print(f"\n{s}: ATAC {a_s.n_obs} → RNA {r_s.n_obs}")

    tp_s, prob = solve_one_sample(s, a_s, r_s, eps=1e-3)

    # if YM2 still doesn't converge, retry once with looser eps
    if not bool(prob.solution.converged):
        print(f"  {s} did NOT converge at eps=1e-3, retrying eps=2e-3")
        tp_s, prob = solve_one_sample(s, a_s, r_s, eps=2e-3)

    T_per_sample[s] = np.asarray(prob.solution.transport_matrix)
    conv[s] = bool(prob.solution.converged)
    print(f"  cost={prob.solution.cost:.4f}  converged={conv[s]}  T={T_per_sample[s].shape}")

In [ ]:
import numpy as np
for s, T in T_per_sample.items():
    Tc = T / T.sum(0, keepdims=True)
    H  = -(Tc * np.log(Tc + 1e-30)).sum(0)
    eff_k = np.exp(H)
    top1  = Tc.max(0)
    print(f"{s}: top1_med={np.median(top1):.3f}  eff_k_med={np.median(eff_k):.0f}  "
          f"converged={conv[s]}  T_sum={T.sum():.4f}")

In [ ]:
out = pathlib.Path("/projects/bhdw/asachan/tmp/T_npy_sample_matched")
out.mkdir(parents=True, exist_ok=True)

for s, T in T_per_sample.items():
    np.save(out / f"T_{s}.npy", T)
    np.save(out / f"src_obs_{s}.npy",
            adata_atac.obs.index[adata_atac.obs["sample"] == s].astype(str).values)
    np.save(out / f"tgt_obs_{s}.npy",
            adata_rna .obs.index[adata_rna .obs["sample"] == s].astype(str).values)

print("artefacts in:", out)

## From OT pairing to peak → gene: barycentric projection + P2G + volcano

The cells below transport ATAC peak accessibility onto every RNA cell using the per-sample GW couplings from the previous section, build the multi-modal `mome` MuData, subset RNA to MaxToki vocab, compute the distance-decay peak→gene matrix `P2G`, and finish with the young-vs-old volcano on per-gene transported peak mass. The last cell saves a single **prep bundle** (`atac_rna_pairing_skm_prep/`) that `maxtoki_multimodal_prep.ipynb` consumes.

In [17]:
# Bridge: ensure T_per_sample / transports / obs_src_per_pair / obs_tgt_per_pair
# are in scope so the barycentric cells below work whether the OT-solve cell
# just ran (T_per_sample already in memory) OR this notebook section is being
# run on its own from the per-sample artifacts saved under T_dir
# (T_{s}.npy + src_obs_{s}.npy + tgt_obs_{s}.npy, written by cell 35).
import os, pathlib, pandas as pd, numpy as np

T_dir = pathlib.Path(os.environ.get(
    "FIREFATE_T_DIR",
    "/projects/bhdw/asachan/tmp/T_npy_sample_matched",
))

if "T_per_sample" not in dir():
    T_per_sample        = {}
    obs_src_per_sample  = {}
    obs_tgt_per_sample  = {}
    for path in sorted(T_dir.glob("T_*.npy")):
        s = path.stem.replace("T_", "")
        T_per_sample[s]       = np.load(path)
        obs_src_per_sample[s] = np.load(T_dir / f"src_obs_{s}.npy", allow_pickle=True)
        obs_tgt_per_sample[s] = np.load(T_dir / f"tgt_obs_{s}.npy", allow_pickle=True)
    if not T_per_sample:
        raise FileNotFoundError(
            f"no T_*.npy artifacts under {T_dir}. Run the OT-solve cell above "
            f"or set FIREFATE_T_DIR to a directory that has them."
        )
    print(f"loaded T_per_sample from {T_dir}: {sorted(T_per_sample)}")
else:
    # OT solve just ran in this kernel -- adata still has the source rows in
    # the same order the solver consumed.
    obs_src_per_sample = {s: adata_atac.obs.index[adata_atac.obs["sample"] == s].astype(str).values
                          for s in T_per_sample}
    obs_tgt_per_sample = {s: adata_rna .obs.index[adata_rna .obs["sample"] == s].astype(str).values
                          for s in T_per_sample}
    print(f"using in-memory T_per_sample: {sorted(T_per_sample)}")

transports       = {(s, s): T_per_sample[s] for s in sorted(T_per_sample)}
obs_src_per_pair = {(s, s): pd.DataFrame(index=pd.Index(np.asarray(obs_src_per_sample[s]).astype(str), name="atac_obs"))
                    for s in sorted(T_per_sample)}
obs_tgt_per_pair = {(s, s): pd.DataFrame(index=pd.Index(np.asarray(obs_tgt_per_sample[s]).astype(str), name="rna_obs"))
                    for s in sorted(T_per_sample)}

human_genome_path = os.environ.get("HUMAN_GENOME_PATH",
                       "/work/hdd/bgdb/asachan/datasets_proj/human_genome_files")
out_tmp           = os.environ.get("OUT_TMP",
                       "/projects/bhdw/asachan/tmp")

print(f"transports: {len(transports)} sample blocks "
      f"({list(transports.keys())})")
print(f"human_genome_path = {human_genome_path}")
print(f"out_tmp           = {out_tmp}")

loaded T_per_sample from /projects/bhdw/asachan/tmp/T_npy_sample_matched: ['OM6', 'OM9', 'YM2']
transports: 3 sample blocks ([('OM6', 'OM6'), ('OM9', 'OM9'), ('YM2', 'YM2')])
human_genome_path = /work/hdd/bgdb/asachan/datasets_proj/human_genome_files
out_tmp           = /projects/bhdw/asachan/tmp


#### Pull soft peak accessibility onto every RNA cell — **sample-stratified** so each RNA cell receives peak mass only from same-donor ATAC. The per-sample GW solves already enforce this on the row blocks; this section assembles the global coupling correctly and adds a defensive sample-mask so cross-sample mass is exactly zero.

In [18]:
import numpy as np, scipy.sparse as sp

# Assemble the global (n_atac, n_rna) coupling by placing each per-sample
# transport block at the correct (row, col) positions on the global grid.
# obs_src_per_pair[(src,tgt)].index / obs_tgt_per_pair[(src,tgt)].index give
# the exact ATAC/RNA cell IDs at the rows/cols of each T block — works whether
# T is compact (n_src_s, n_tgt_s) or full-width.
atac_pos = {idx: i for i, idx in enumerate(adata_atac.obs.index.astype(str))}
rna_pos  = {idx: i for i, idx in enumerate(adata_rna .obs.index.astype(str))}

T_full = np.zeros((adata_atac.n_obs, adata_rna.n_obs), dtype=np.float32)
for (src_b, tgt_b), T in transports.items():
    src_obs = obs_src_per_pair[(src_b, tgt_b)].index.astype(str)
    tgt_obs = obs_tgt_per_pair[(src_b, tgt_b)].index.astype(str)
    if T.shape != (len(src_obs), len(tgt_obs)):
        raise ValueError(
            f"{src_b}->{tgt_b}: T {T.shape} != ({len(src_obs)},{len(tgt_obs)})"
        )
    src_i = np.fromiter((atac_pos[i] for i in src_obs), dtype=np.int64, count=len(src_obs))
    tgt_i = np.fromiter((rna_pos [i] for i in tgt_obs), dtype=np.int64, count=len(tgt_obs))
    T_full[np.ix_(src_i, tgt_i)] = T.astype(np.float32)
    print(f"  placed {src_b} -> {tgt_b}  block={T.shape}")

# SAMPLE-stratify (defensive) — ensures each RNA cell receives peak mass ONLY
# from ATAC cells of the *same donor*.  Per-sample GW solves already enforce
# this; this step is a no-op when the export is clean and a hard guard
# otherwise.  This is what makes a query cell's chromatin bias come from its
# own donor's ATAC even when the perturb pipeline pools across donors of the
# same age (e.g. all O80 cells from donor A and donor B).
atac_sample = adata_atac.obs["sample"].astype(str).values
rna_sample  = adata_rna .obs["sample"].astype(str).values

sample_match = atac_sample[:, None] == rna_sample[None, :]
n_xs = int(((T_full > 0) & ~sample_match).sum())
T_strat = np.where(sample_match, T_full, 0.0).astype(np.float32)
print(f"  zeroed {n_xs} cross-sample T entries (should be 0 if OT was per-sample)")

col_sum  = T_strat.sum(0, keepdims=True)
n_orphan = int((col_sum.ravel() == 0).sum())
print(f"  RNA cells with no same-sample ATAC source: {n_orphan} / {T_strat.shape[1]}")
T_col    = T_strat / np.where(col_sum > 0, col_sum, 1.0)        # column-stochastic

# soft peak matrix per RNA cell (raw Tn5 counts, not TF-IDF)
X_peak      = adata_atac.X.toarray() if sp.issparse(adata_atac.X) else adata_atac.X
X_peaks_rna = T_col.T @ X_peak                                   # (n_rna, n_peaks)

adata_rna.obsm["X_peaks_inferred"] = X_peaks_rna.astype(np.float32)
print("inferred peaks per RNA cell:", X_peaks_rna.shape,
      "  mean nnz / cell:", float((X_peaks_rna > 0).sum(1).mean()))

  placed OM6 -> OM6  block=(243, 722)
  placed OM9 -> OM9  block=(2238, 1278)
  placed YM2 -> YM2  block=(3349, 1989)
  zeroed 0 cross-sample T entries (should be 0 if OT was per-sample)
  RNA cells with no same-sample ATAC source: 0 / 3989
inferred peaks per RNA cell: (3989, 27649)   mean nnz / cell: 26290.237653547254


In [19]:
import pandas as pd
# Mass each RNA cell receives, grouped by donor sample.  Within a donor we
# expect roughly uniform mean mass; orphans (mass == 0) should be 0 by
# construction since each sample has its own ATAC cells. (om6 just has low rna cells and this mean depends on 1/n_of_cells)
print(pd.DataFrame({
    "rna_sample":    rna_sample,
    "received_mass": col_sum.ravel(),
}).groupby("rna_sample")["received_mass"].agg(["mean", "std", "min", "max", "count"]))

                mean       std       min       max  count
rna_sample                                               
OM6         0.001385  0.000002  0.001384  0.001392    722
OM9         0.000782  0.000002  0.000752  0.000786   1278
YM2         0.000503  0.000004  0.000502  0.000599   1989


#### is the transition probability too diffused or does it still capture geometrical structure above uniform coupling?
#### also fewer atac-neighbours

In [20]:
import numpy as np, pandas as pd
T_col_n = T_strat / np.where(col_sum > 0, col_sum, 1.0)
H       = -(T_col_n * np.log(T_col_n + 1e-30)).sum(0)
df = pd.DataFrame({
    "sample": rna_sample,
    "top1":   T_col_n.max(0),
    "eff_k":  np.exp(H),
})
print(df.groupby("sample").agg(["median", lambda x: np.percentile(x, 95)]))

            top1                 eff_k            
          median <lambda_0>     median  <lambda_0>
sample                                            
OM6     0.598465   0.999960   3.167665   30.764732
OM9     0.181309   0.537463  27.169317  103.766083
YM2     0.106789   0.501286  63.279961  209.356628


## Multi-ome data

In [21]:
# ---- (cell 20 replacement) build MuData from raw soft peaks -----------
import mudata as mu, anndata as ad

peaks    = adata_atac.var_names.to_list()
peak_var = adata_atac.var[["chrom","start","end"]].copy() \
           if {"chrom","start","end"}.issubset(adata_atac.var.columns) else None
if peak_var is None:
    parsed = [p.replace(":", "-").split("-") for p in peaks]
    peak_var = pd.DataFrame(parsed, columns=["chrom","start","end"], index=peaks) \
                 .astype({"start": int, "end": int})

rna  = ad.AnnData(
    X=adata_rna.X if sp.issparse(adata_rna.X) else sp.csr_matrix(adata_rna.X),
    obs=adata_rna.obs.copy(), var=adata_rna.var.copy(),
    obsm={"X_pca": adata_rna.obsm["X_pca"]},
    layers={k: v.copy() for k, v in adata_rna.layers.items()},  # carry counts (and any other layers) through
)
atac = ad.AnnData(
    X=sp.csr_matrix(X_peaks_rna),
    obs=adata_rna.obs.copy(), var=peak_var,
)
atac.var_names = peaks
mome = mu.MuData({"rna": rna, "atac": atac})

# ---- (cell 21 replacement) TF-IDF in-place on mome ATAC ---------------
def tfidf(M, scale=1e4):
    M = M.tocsr() if sp.issparse(M) else sp.csr_matrix(M)
    rs  = np.asarray(M.sum(1)).ravel() + 1e-12
    cs  = np.asarray(M.sum(0)).ravel() + 1e-12
    idf = np.log1p(M.shape[0] / cs)
    Mn  = sp.diags(scale / rs) @ M @ sp.diags(idf)
    return Mn.log1p()

mome["atac"].layers["tfidf"] = tfidf(mome["atac"].X)
print(mome)

/projects/bhdw/asachan/.conda/envs/moscot/lib/python3.11/site-packages/mudata/_core/mudata.py:1416: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/projects/bhdw/asachan/.conda/envs/moscot/lib/python3.11/site-packages/mudata/_core/mudata.py:1272: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)


MuData object with n_obs × n_vars = 3989 × 76004
  2 modalities
    rna:	3989 × 48355
      obs:	'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'sample', 'percent.mt', 'age', 'tech', 'Sex', 'Country', 'age_pop', 'Annotation', 'Pseudotime', 'Pseudotime_typeI', 'Pseudotime_typeII', 'bead_count', 'age_categorical', 'GOBP_DNA_DAMAGE_RESPONSE', 'GOBP_DNA_REPAIR', 'HALLMARK_DNA_REPAIR', 'REACTOME_DNA_REPAIR'
      var:	'features', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
      obsm:	'X_pca'
      layers:	'counts', 'counts_float', 'lognorm'
    atac:	3989 × 27649
      obs:	'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'sample', 'percent.mt', 'age', 'tech', 'Sex', 'Country', 'age_pop', 'Annotation', 'Pseudotime', 'Pseudotime_typeI', 'Pseudotime_typeII', 'bead_count', 'age_categorical', 'GOBP_DNA_DAMAGE_RESPONSE', 'GOBP_DNA_REPAIR', 'HALLMARK_DNA_REPAIR', 'REACTOME_DNA_REPAIR'
      var:	'chrom', 'start', 'end'
      layers:	'tfidf'


In [22]:
# save the MuData object to a file
out_file = pathlib.Path(objects_dir) / "mome_rna_atac_inferred_peaks.h5mu"
mome.write_h5mu(out_file)
print("saved MuData to:", out_file)

saved MuData to: /work/hdd/bgdb/asachan/datasets_proj/SKM_ageing_human/mome_rna_atac_inferred_peaks.h5mu


/projects/bhdw/asachan/.conda/envs/moscot/lib/python3.11/site-packages/mudata/_core/mudata.py:1416: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/projects/bhdw/asachan/.conda/envs/moscot/lib/python3.11/site-packages/mudata/_core/mudata.py:1272: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)


## Subset the rna to MaxToki vocab

In [49]:
import pandas as pd, pyranges as pr, numpy as np

# 1. symbol → ENSG from the same GTF you used for TSS lookup
gtf = pr.read_gtf(f"{human_genome_path}/gencode.v46.annotation.gtf").df
gtf = gtf[gtf.Feature == "gene"][["gene_name", "gene_id"]].drop_duplicates("gene_name")
gtf["gene_id_clean"] = gtf["gene_id"].str.split(".").str[0]    # strip ENSG version
sym2ensg = dict(zip(gtf["gene_name"].astype(str), gtf["gene_id_clean"]))
print(f"GTF mappings: {len(sym2ensg)} symbol→ENSG")

# 2. attach ENSG to mome["rna"].var
syms = mome["rna"].var_names.astype(str)
mome["rna"].var["ensg"] = [sym2ensg.get(s, None) for s in syms]

n_total   = mome["rna"].n_vars
n_mapped  = mome["rna"].var["ensg"].notna().sum()
print(f"{n_mapped}/{n_total} symbols mapped to ENSG  ({n_mapped/n_total:.1%})")

GTF mappings: 61471 symbol→ENSG
29718/48355 symbols mapped to ENSG  (61.5%)


In [50]:
import json, numpy as np, pandas as pd

# 1. load MaxToki vocab
with open("/projects/bhdw/asachan/methods/maxtoki-perturb/src/maxtoki_mlx/resources/token_dictionary.json") as f:
    vocab = json.load(f)

# strip special tokens to leave only ENSG entries
ensg_to_token = {k: v for k, v in vocab.items() if k.startswith("ENSG")}
print(f"vocab tokens: {len(vocab)} total, {len(ensg_to_token)} ENSG-keyed")

# 2. attach token index to mome["rna"].var
mome["rna"].var["maxtoki_token"] = (
    mome["rna"].var["ensg"].map(ensg_to_token).astype("Int64")  # nullable int
)

n_in_vocab = mome["rna"].var["maxtoki_token"].notna().sum()
print(f"vars with MaxToki token: {n_in_vocab} / {mome['rna'].n_vars} "
      f"({n_in_vocab / mome['rna'].n_vars:.1%})")

vocab tokens: 20277 total, 20271 ENSG-keyed
vars with MaxToki token: 18470 / 48355 (38.2%)


In [51]:
import numpy as np

mask = mome["rna"].var["maxtoki_token"].notna()
print(f"keeping {mask.sum()} / {mome['rna'].n_vars} genes")

# rebuild MuData with subsetted RNA modality, ATAC unchanged
import mudata as mu
rna_sub = mome["rna"][:, mask.values].copy()
rna_sub.var["maxtoki_token"] = rna_sub.var["maxtoki_token"].astype(int)
rna_sub = rna_sub[:, np.argsort(rna_sub.var["maxtoki_token"].values)].copy()  # sort by token id


keeping 18470 / 48355 genes


In [52]:
rna_sub.var["ensembl_id"] = rna_sub.var['ensg']        # if ENSGs are the index

In [53]:
mome["rna"].var["ensembl_id"] = mome["rna"].var['ensg']        # if ENSGs are the index

In [54]:
# rna_sub already inherits adata_rna's layers (counts, anything else) because
# we passed layers=... when building the rna AnnData two cells up, and AnnData
# slicing/reordering in the rna_sub creation cell carries layers along. Verify
# and rebuild mome with the vocab-subset RNA.
import mudata as mu

assert "counts" in rna_sub.layers, (
    "rna_sub.layers['counts'] is missing. Check that the 'ensure counts layer' "
    "cell ran after the RNA load AND that the mome construction cell passes "
    "layers=... when building the rna AnnData."
)
print(f"[ok] rna_sub layers preserved: {list(rna_sub.layers.keys())}")

mome = mu.MuData({"rna": rna_sub, "atac": mome["atac"]})
print(mome)

[ok] rna_sub layers preserved: ['counts', 'counts_float', 'lognorm']
MuData object with n_obs × n_vars = 3989 × 46119
  2 modalities
    rna:	3989 × 18470
      obs:	'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'sample', 'percent.mt', 'age', 'tech', 'Sex', 'Country', 'age_pop', 'Annotation', 'Pseudotime', 'Pseudotime_typeI', 'Pseudotime_typeII', 'bead_count', 'age_categorical', 'GOBP_DNA_DAMAGE_RESPONSE', 'GOBP_DNA_REPAIR', 'HALLMARK_DNA_REPAIR', 'REACTOME_DNA_REPAIR'
      var:	'features', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'ensg', 'maxtoki_token', 'ensembl_id'
      obsm:	'X_pca'
      layers:	'counts', 'counts_float', 'lognorm'
    atac:	3989 × 27649
      obs:	'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'sample', 'percent.mt', 'age', 'tech', 'Sex', 'Country', 'age_pop', 'Annotation', 'Pseudotime', 'Pseudotime_typeI', 'Pseudotime_typeII', 'bead_count', 'age_categorical', 'GOBP_DNA_DAMAGE_RESPONSE', 'GOBP_DNA_REPAIR', 'HALLMARK_DNA_REPAIR', 'REACTOME_DNA_RE

/projects/bhdw/asachan/.conda/envs/moscot/lib/python3.11/site-packages/mudata/_core/mudata.py:1416: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/projects/bhdw/asachan/.conda/envs/moscot/lib/python3.11/site-packages/mudata/_core/mudata.py:1272: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)


In [55]:
#write mudata to file
mome.write_h5mu('multiome_atac_rna.h5mu')

/projects/bhdw/asachan/.conda/envs/moscot/lib/python3.11/site-packages/mudata/_core/mudata.py:1416: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/projects/bhdw/asachan/.conda/envs/moscot/lib/python3.11/site-packages/mudata/_core/mudata.py:1272: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)
